# 02 - Data Cleaning

**Objective:** Address the issues identified during the Data Understanding stage to create a clean dataset ready for modeling.

> **Principle:** Only handle the issues that have been identified. As confirmed in NB01: there are no missing NaN values and no duplicate records.

**Steps:**
1. Load and sort the data chronologically
2. Remove unnecessary columns
3. Scale numerical variables by Match Week (MW)
4. Apply One-Hot Encoding for HM1–HM3 and AM1–AM3
5. Encode the target variable FTR
6. Perform final checks and save the dataset

## 1. Import & Load data

In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw_data.csv', index_col=0)
print(f'Original shape: {df.shape}')
df.head(3)

Original shape: (6840, 39)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTGS,ATGS,HTGC,ATGC,...,HTLossStreak3,HTLossStreak5,ATWinStreak3,ATWinStreak5,ATLossStreak3,ATLossStreak5,HTGD,ATGD,DiffPts,DiffFormPts
0,19/08/00,Charlton,Man City,4,0,H,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,19/08/00,Chelsea,West Ham,4,2,H,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0
2,19/08/00,Coventry,Middlesbrough,1,3,NH,0,0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0,0.0


## 2. Sort data by time

> Sorting data in chronological order is mandatory before splitting into train/test sets. If the data is not properly sorted, splitting by index will not preserve temporal order, which can lead to data leakage.

In [14]:
# Convert Date to datetime and sort
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

print(f'First match: {df["Date"].iloc[0].date()}')
print(f'Last match: {df["Date"].iloc[-1].date()}')
print(f'Total matches: {len(df)}')

First match: 2000-08-19
Last match: 2018-05-13
Total matches: 6840


## 3. Remove unnecessary columns

In [15]:
cols_to_drop = [
    # Identifiers — no predictive value
    'HomeTeam', 'AwayTeam',
    # String format 'WWDLL' — already aggregated into numbers in HTFormPts, ATFormPts
    'HTFormPtsStr', 'ATFormPtsStr',
    # DATA LEAKAGE: FTHG, FTAG are match results — known before prediction, constitutes data leakage
    'FTHG', 'FTAG',
    # REDUNDANT: HTGS, ATGS, HTGC, ATGC have been aggregated into HTGD (= HTGS - HTGC)
    # and indirectly into HTP, ATP — keeping them causes multicollinearity
    'HTGS', 'ATGS', 'HTGC', 'ATGC',
    # LOW INFORMATION VALUE: results from 4–5 matches ago are less correlated than the most recent 3 matches
    'HM4', 'HM5', 'AM4', 'AM5'
]

df = df.drop(columns=cols_to_drop)
print(f'Removed {len(cols_to_drop)} columns.')
print(f'Shape after dropping columns: {df.shape}')
print(f'\nRemaining columns: {list(df.columns)}')

Removed 14 columns.
Shape after dropping columns: (6840, 25)

Remaining columns: ['Date', 'FTR', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'AM1', 'AM2', 'AM3', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


**Detailed explanation of each group:**

| Group | Columns | Reason for removal |
|:---|:---|:---|
| Identification | `HomeTeam`, `AwayTeam` | Do not provide direct predictive information |
| Redundant | `HTFormPtsStr`, `ATFormPtsStr` | Numeric versions `HTFormPts`, `ATFormPts` already exist |
| **Data leakage** | `FTHG`, `FTAG` | Match results not available at prediction time |
| Redundant | `HTGS`, `ATGS`, `HTGC`, `ATGC` | Already aggregated into `HTGD` = HTGS − HTGC, and indirectly reflected in `HTP`/`ATP` |
| Low information | `HM4`, `HM5`, `AM4`, `AM5` | Results from 4–5 previous matches have weaker correlation than the last 3 matches |

> **Keep `Date`** at this stage for time-based train/test splitting in NB03. It will be dropped before feeding into the model.

## 4. Scale numerical variables by Match Week (MW)

> **Why is scaling by MW necessary?**
>
> Cumulative variables such as `HTP`, `HTGD`, `DiffPts`, etc. naturally increase over the course of a season:
> - **30 points at Week 10** → ~3 points per match (very strong performance)
> - **30 points at Week 30** → ~1 point per match (only average performance)
>
> → Therefore, dividing by MW converts these features into per-match average indicators, ensuring fair comparability across different stages of the season.

In [16]:
scale_cols = ['HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'HTP', 'ATP']
df['MW'] = df['MW'].astype(float)

print('Example before scaling (week 10):')
print(df[df['MW'] == 10][scale_cols + ['MW']].head(2).to_string())

for col in scale_cols:
    df[col] = df[col] / df['MW']

print('\nExample after scaling (week 10) — average values per match:')
print(df[df['MW'] == 10][scale_cols].head(2).to_string())

# Drop MW after using it
df = df.drop(columns=['MW'])
print('\n Scale completed. Column MW dropped.')

Example before scaling (week 10):
    HTGD  ATGD  DiffPts  DiffFormPts  HTP  ATP    MW
89  -0.1   0.6     -1.0         -0.5  0.8  1.8  10.0
90  -0.1  -0.9      0.6          0.2  1.1  0.5  10.0

Example after scaling (week 10) — average values per match:
    HTGD  ATGD  DiffPts  DiffFormPts   HTP   ATP
89 -0.01  0.06    -0.10        -0.05  0.08  0.18
90 -0.01 -0.09     0.06         0.02  0.11  0.05

 Scale completed. Column MW dropped.


## 5. One-Hot Encoding for HM1–HM3, AM1–AM3

> **Why use One-Hot Encoding instead of W=3 / D=1 / L=0?**
>
> W/D/L are nominal variables — they do not have a true mathematical linear order. If encoded as W=3, D=1, L=0, the model assumes that the distance from W→D is equal to D→L = 1, which is semantically incorrect.
>
> One-Hot Encoding creates independent binary columns, allowing the model to learn separate weights for each outcome.
> `'M'` (insufficient match history) is kept as a separate category instead of being forced into 0 (loss).

In [17]:
result_cols_ohe = ['HM1', 'HM2', 'HM3', 'AM1', 'AM2', 'AM3']

print('Unique values in HM1:', sorted(df['HM1'].unique()))

for col in result_cols_ohe:
    df[col] = df[col].astype(str)

df = pd.get_dummies(df, columns=result_cols_ohe, prefix=result_cols_ohe)

ohe_cols = [c for c in df.columns if any(c.startswith(p+'_') for p in result_cols_ohe)]
print(f'\nNew OHE columns ({len(ohe_cols)} columns):')
print(ohe_cols)
print(f'\nShape after OHE: {df.shape}')

Unique values in HM1: ['D', 'L', 'M', 'W']

New OHE columns (24 columns):
['HM1_D', 'HM1_L', 'HM1_M', 'HM1_W', 'HM2_D', 'HM2_L', 'HM2_M', 'HM2_W', 'HM3_D', 'HM3_L', 'HM3_M', 'HM3_W', 'AM1_D', 'AM1_L', 'AM1_M', 'AM1_W', 'AM2_D', 'AM2_L', 'AM2_M', 'AM2_W', 'AM3_D', 'AM3_L', 'AM3_M', 'AM3_W']

Shape after OHE: (6840, 42)


**Observation:** 6 columns × 4 values (W/D/L/M) = 24 new binary columns. The model can learn the separate impact of each recent match outcome.

## 6. Encode the target variable FTR

In [18]:
df['FTR'] = df['FTR'].map({'H': 1, 'NH': 0})
print('Distribution of FTR after encoding:')
print(df['FTR'].value_counts())
print(f'\nHome Win Rate: {df["FTR"].mean()*100:.1f}%')

Distribution of FTR after encoding:
FTR
0    3664
1    3176
Name: count, dtype: int64

Home Win Rate: 46.4%


## 🔹 7. Final result check

In [19]:
print('=== INFORMATION AFTER CLEANING ===')
print(f'Shape:          {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('All columns:')
print(list(df.columns))
print()
df.head(3)

=== INFORMATION AFTER CLEANING ===
Shape:          (6840, 42)
Missing values: 0
Duplicate rows: 113

All columns:
['Date', 'FTR', 'HTP', 'ATP', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts', 'HM1_D', 'HM1_L', 'HM1_M', 'HM1_W', 'HM2_D', 'HM2_L', 'HM2_M', 'HM2_W', 'HM3_D', 'HM3_L', 'HM3_M', 'HM3_W', 'AM1_D', 'AM1_L', 'AM1_M', 'AM1_W', 'AM2_D', 'AM2_L', 'AM2_M', 'AM2_W', 'AM3_D', 'AM3_L', 'AM3_M', 'AM3_W']



,Date,FTR,HTP,ATP,HTFormPts,ATFormPts,HTWinStreak3,HTWinStreak5,HTLossStreak3,HTLossStreak5,...,AM1_M,AM1_W,AM2_D,AM2_L,AM2_M,AM2_W,AM3_D,AM3_L,AM3_M,AM3_W
0,2000-08-19,1,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False
1,2000-08-19,1,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False
2,2000-08-19,0,0.0,0.0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,False


## 8. Save the cleaned dataset

In [20]:
df.to_csv('../data/cleaned_data.csv', index=False)
print('Saved: ../data/cleaned_data.csv')
print(f'   Shape: {df.shape}')
print(f'   Includes: Date (for train/test split), FTR (target Classification), and {len(df.columns)-2} features')

Saved: ../data/cleaned_data.csv
   Shape: (6840, 42)
   Includes: Date (for train/test split), FTR (target Classification), and 40 features


## Summary

| Step | Action | Result |
|:---|:---|:---|
| Sort by time | `pd.to_datetime` + `sort_values('Date')` | Ensures correct order for time-based split |
| Drop columns | 14 columns (leakage + redundant + low-information) | 39 → 25 columns (before OHE) |
| Scale by MW | Divide 6 cumulative variables by MW | Normalized across season stages |
| One-Hot Encoding | HM1–HM3, AM1–AM3 → 24 binary columns | Proper encoding for nominal variables |
| Encode FTR | H=1, NH=0 | Classification target variable |

**Conclusion:** Clean dataset contains **6,840 rows × 43 columns** (including `Date` and `FTR`), ready for modeling.